In [4]:
import sys
sys.path.append("..")
import ee
ee.Authenticate()
ee.Initialize(project="tree-fire-510209")

from src.fires import load_fire

hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
print(hansen.bandNames().getInfo())

uid = "2016_7022"
fire = load_fire(uid)
aoi = ee.Geometry.Rectangle(list(fire.total_bounds))

lossyear = hansen.select("lossyear").clip(aoi)
hist = lossyear.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi, scale=30, maxPixels=1e9,
).getInfo()
print(uid, "lossyear pixel counts (0 = no loss detected, else 2000+N):", hist)

['treecover2000', 'loss', 'gain', 'lossyear', 'first_b30', 'first_b40', 'first_b50', 'first_b70', 'last_b30', 'last_b40', 'last_b50', 'last_b70', 'datamask']
2016_7022 lossyear pixel counts (0 = no loss detected, else 2000+N): {'lossyear': {'1': 205.47450980392156, '10': 1233.9803921568628, '11': 2316.5058823529407, '12': 991.7843137254904, '13': 1570.5882352941176, '14': 2200.972549019608, '15': 2871.364705882352, '16': 342203.34509803925, '17': 264276.06666666653, '18': 684, '19': 519, '2': 5592.882352941175, '20': 260, '21': 192, '22': 346, '23': 222, '24': 149143.9019607854, '25': 14894.223529411804, '3': 162.49803921568628, '4': 522.3333333333333, '5': 67.87450980392157, '6': 873.6941176470589, '7': 4157.776470588233, '8': 556.2941176470588, '9': 539}}


In [2]:
import ee

hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")

def lossyear_histogram(geom):
    img = hansen.select("lossyear").clip(geom)
    return img.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=geom, scale=30, maxPixels=1e9,
    ).getInfo()["lossyear"]

for uid in ["2016_7022", "2018_7021"]:
    fire = load_fire(uid)
    geom = fire.geometry.iloc[0]
    fire_aoi = ee.Geometry(geom.__geo_interface__)

    # nearby control: a ring outside the fire perimeter, not touching it directly
    control_geom = geom.buffer(0.15).difference(geom.buffer(0.03))
    control_aoi = ee.Geometry(control_geom.__geo_interface__)

    fire_hist = lossyear_histogram(fire_aoi)
    control_hist = lossyear_histogram(control_aoi)

    print(f"=== {uid}: FIRE AREA ===")
    print(fire_hist)
    print(f"=== {uid}: CONTROL (nearby, unburned) ===")
    print(control_hist)
    print()

=== 2016_7022: FIRE AREA ===
{'1': 15.533333333333333, '10': 66.74901960784314, '11': 184.84705882352938, '12': 745.070588235294, '13': 74.45098039215686, '14': 182.85882352941178, '15': 588.1333333333334, '16': 264888.1098039218, '17': 207510.4156862747, '18': 465.8313725490196, '19': 308.3450980392157, '2': 15.070588235294117, '20': 49, '21': 8.501960784313725, '22': 15.901960784313726, '23': 59.356862745098034, '24': 3067.0470588235326, '25': 516.8980392156864, '3': 30, '4': 160.77254901960785, '5': 15, '6': 18, '7': 34, '8': 43.47450980392157, '9': 34.05490196078431}
=== 2016_7022: CONTROL (nearby, unburned) ===
{'1': 1376, '10': 12110.811764705883, '11': 16448.949019607844, '12': 5366.396078431372, '13': 11164.76862745098, '14': 25631.05882352941, '15': 5780.815686274509, '16': 101290.54901960777, '17': 99738.21176470595, '18': 23185.227450980394, '19': 8103.219607843137, '2': 13207.141176470588, '20': 2327, '21': 2237.41568627451, '22': 1980.556862745098, '23': 2932.0823529411764